In [ ]:
# 1. Google Drive'ı bağla (Checkpoints doğrudan Drive'a yazılacak)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/vesivuus_checkpoints', exist_ok=True)

# 2. Gerekli kütüphaneleri kur
!pip install -q monai tifffile imagecodecs kagglehub

In [ ]:
import os

# Sol panele sürüklediğin kaggle.json'ı sistem yerine kopyala
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kagglehub

# Yarışma verisini indir
path = kagglehub.competition_download("vesuvius-challenge-surface-detection")
print("[+] Dataset başarıyla indirildi, dizin:", path)

# Sembolik bağ ve kod paketini aç
!rm -rf dataset
!ln -s "{path}" dataset

!unzip -o -q /content/vesivuus_colab.zip -d /content/
print("[+] Kodlar ve modeller başarıyla hazırlandı!")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -u parametresi çıktıyı anlık ekrana basar, %%bash kaldırıldığı için genel yüzde canlı akar
!python -u train.py \
  --arch segresnet \
  --patch_size 192 \
  --batch_size 1 \
  --accum_steps 2 \
  --samples_per_volume 1 \
  --all_data \
  --epochs 160 \
  --lr 1e-4 \
  --patience 9999 \
  --save_interval 25 \
  --pretrained checkpoints/segresnet3d_p160_fold2_best.pth \
  --save_dir /content/drive/MyDrive/vesivuus_checkpoints \
  --tag finetune192 \
  --resume